# dLLM Quantization Experiment Pipeline
**환경**: Colab Pro+ A100 (VS Code remote 연결)

**순서**:
1. GPU 확인
2. Google Drive 마운트
3. GitHub repo clone
4. 의존성 설치
5. 모델 경로 확인
6. 실험 실행

## Cell 1: GPU 확인 + Drive 마운트 + Repo Clone

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 1. GPU 확인
# ═══════════════════════════════════════════════════════════════
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

: 

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2. Google Drive 마운트
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# 모델 존재 확인
import os
MODEL_PATH = '/content/drive/MyDrive/dLLM_settings/dLLM/models/LLaDA-8B-Instruct'

assert os.path.exists(MODEL_PATH), f"모델 없음: {MODEL_PATH}"
model_files = os.listdir(MODEL_PATH)
safetensor_files = [f for f in model_files if f.endswith('.safetensors')]
print(f"✅ 모델 확인: {len(safetensor_files)} safetensor files")
print(f"   경로: {MODEL_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3. GitHub repo clone (Colab 리셋 시에도 동작)
# ═══════════════════════════════════════════════════════════════
REPO_URL = 'https://github.com/CH0901/dLLM.git'
WORK_DIR = '/content/dLLM'

if os.path.exists(WORK_DIR):
    print("Repo already exists, pulling latest...")
    !cd {WORK_DIR} && git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {WORK_DIR}

print(f"\n✅ Repo ready at {WORK_DIR}")
!ls {WORK_DIR}/dllm_quant/

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 4. 의존성 설치
# ═══════════════════════════════════════════════════════════════
!pip install -q datasets transformers accelerate sentencepiece protobuf

## Cell 2: 실험 실행

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5. 실험 실행: GPTAQ W4A16 (양자화만, eval 스킵)
# ═══════════════════════════════════════════════════════════════
%cd /content/dLLM/dllm_quant

!python run_experiment.py \
    --method gptaq \
    --weight_bits 4 \
    --act_bits 16 \
    --model_path {MODEL_PATH} \
    --n_cal_samples 16 \
    --n_eval_samples 2 \
    --skip_eval \
    --output_dir /content/dLLM/results

## (Optional) 추가 실험들

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GPTAQ W3A16
# ═══════════════════════════════════════════════════════════════
# !python run_experiment.py \
#     --method gptaq \
#     --weight_bits 3 \
#     --act_bits 16 \
#     --model_path {MODEL_PATH} \
#     --n_cal_samples 128 \
#     --skip_eval \
#     --output_dir /content/dLLM/results

In [ ]:
# ═══════════════════════════════════════════════════════════════
# QuaRot + GPTAQ W4A16
# ═══════════════════════════════════════════════════════════════
# !python run_experiment.py \
#     --method quarot+gptaq \
#     --weight_bits 4 \
#     --act_bits 16 \
#     --model_path {MODEL_PATH} \
#     --n_cal_samples 128 \
#     --skip_eval \
#     --output_dir /content/dLLM/results

In [ ]:
# ═══════════════════════════════════════════════════════════════
# FP vs Quant 분석 메트릭 (eval 포함)
# ═══════════════════════════════════════════════════════════════
# !python run_experiment.py \
#     --method gptaq \
#     --weight_bits 4 \
#     --act_bits 16 \
#     --model_path {MODEL_PATH} \
#     --n_cal_samples 128 \
#     --n_eval_samples 10 \
#     --output_dir /content/dLLM/results

## Cell 3: 결과 확인

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 결과 확인
# ═══════════════════════════════════════════════════════════════
import json

results_dir = '/content/dLLM/results'
for exp_name in os.listdir(results_dir):
    exp_path = os.path.join(results_dir, exp_name)
    if not os.path.isdir(exp_path):
        continue
    
    config_path = os.path.join(exp_path, 'config.json')
    summary_path = os.path.join(exp_path, 'summary.json')
    model_path = os.path.join(exp_path, 'quantized_model')
    
    print(f"\n{'='*50}")
    print(f"Experiment: {exp_name}")
    print(f"{'='*50}")
    
    if os.path.exists(config_path):
        with open(config_path) as f:
            print(f"Config: {json.load(f)}")
    
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            print(f"Summary: {json.dumps(json.load(f), indent=2)}")
    
    if os.path.exists(model_path):
        model_files = os.listdir(model_path)
        print(f"Quantized model saved: {len(model_files)} files")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# (Optional) 결과를 Drive에 백업
# ═══════════════════════════════════════════════════════════════
# BACKUP_DIR = '/content/drive/MyDrive/dLLM_settings/dLLM/results'
# !cp -r /content/dLLM/results/* {BACKUP_DIR}/